# NB-02c — clean WER for `ckpt-xlsr-1h-traj/final`

**~2 GPU-minutes. One checkpoint, 100 clips, no test set, no FLEURS.**

## Why this exists

The checkpoint-variance control (NB-12) showed that a second training run of the 1-hour rung is
attacked at **41.0%** where the first is attacked at **75.0%** — a 34 pp gap from training-run
variance alone. To put that condition into the pooled ANCOVA, its **clean WER on the same 100
utterances** is needed as the covariate, and it has never been measured.

It matters more than a missing cell usually would. With the WER estimated at 0.659 the pooled
coefficient is $\beta=+0.108$ with a 95% CI of $[+0.008, +0.208]$ — it excludes zero. At a
plausible lower value of 0.600 the interval becomes $[-0.018, +0.208]$ and **includes zero**.
So the significance of the paper's central estimate currently rests on a number nobody measured.

This measures it.

## Attach

`mahdihasanqurishi/ckpt-xlsr-1h-traj`, `mahdihasanqurishi/bn-asr-splits`, `bengaliai-speech`,
and any one `zoayriaabedin/eps50-*-n100` (for the frozen 100-utterance list).

GPU T4 x2. Internet off is fine — nothing here needs the pinned transformers, and the pin will
fail silently as it did in the last two runs. The version is recorded either way.

## 1 · Config

In [ ]:
CKPT_DATASET = "ckpt-xlsr-1h-traj"
CHECKPOINT   = "final"
MAX_DUR      = 10.0
EVAL_BS      = 8
SEED         = 1337

import os, sys, json, time, re, glob, random, unicodedata, hashlib
import numpy as np, pandas as pd, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK, DEVICE = "/kaggle/working", "cuda"
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2"
print(f"NB-02c | {CKPT_DATASET}/{CHECKPOINT} | clean WER on the 100-utterance attack set")

## 2 · Pins

In [ ]:
# Internet is off in this session, so `pip install transformers==4.44.2` cannot
# reach the index. Measured twice in this project's own logs (NB-12 and NB-13,
# 2026-09-30): the cell spends ~202 s timing out, prints two ERROR lines, fails,
# and leaves the preinstalled version in place anyway. Pure waste.
#
# So we do not attempt the pin. We import what is already present and RECORD the
# version instead -- the pin's purpose was reproducibility, and a recorded
# version serves that better than a failed install does.
#
# If internet is ever turned ON, restore the pin: 4.44.2 is the version every
# figure measured before 2026-09-30 was produced on, and mixing versions across
# figures is exactly what the provenance record exists to catch.
#
# NOTE: that cell also installed `jiwer`, which is NOT preinstalled on Kaggle.
# WER/CER are therefore implemented inline below, verified against jiwer to
# machine precision. transformers, librosa and numpy ARE preinstalled.
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import transformers, librosa, numpy
print(f"transformers {transformers.__version__} | numpy {numpy.__version__} "
      f"| librosa {librosa.__version__}")
if transformers.__version__ != "4.44.2":
    print(f"  NOTE: this is NOT the project pin (4.44.2). Record "
          f"{transformers.__version__} against every number this run produces.")


## 3 · Inputs

In [ ]:
def list_inputs():
    out = []
    for pat in ("/kaggle/input/*", "/kaggle/input/*/*", "/kaggle/input/*/*/*"):
        out += [p for p in glob.glob(pat) if os.path.isdir(p)]
    return sorted(set(out))

INPUTS_LIST = list_inputs()

def find_input(pattern, what, required=True):
    hits = [p for p in INPUTS_LIST if re.search(pattern, os.path.basename(p), re.I)]
    if not hits:
        if not required: return None
        print("Visible under /kaggle/input:"); [print("   ", p) for p in INPUTS_LIST]
        raise AssertionError(f"Could not find {what} (regex {pattern})")
    return max(hits, key=lambda p: (p.count("/"), len(p)))

# NB-02b's model-discovery loop is deliberately NOT carried over: it iterates
# RUNGS over ckpt-xlsr-{r}h, and this notebook resolves exactly one
# checkpoint, by name, in the next cell.


## 4 · Resolve the checkpoint and the frozen 100

In [ ]:
cands = [p for p in INPUTS_LIST if re.search(rf"{re.escape(CKPT_DATASET)}$", os.path.basename(p))]
assert len(cands) == 1, (f"need exactly one input matching '{CKPT_DATASET}', got "
                         f"{[os.path.basename(c) for c in cands]}")
CKPT = cands[0]
MODEL_DIR = CKPT if os.path.exists(f"{CKPT}/config.json") else f"{CKPT}/final"
assert os.path.exists(f"{MODEL_DIR}/config.json"), f"no config.json under {MODEL_DIR}"
print("checkpoint:", MODEL_DIR)

SPLITS = next((p for p in INPUTS_LIST if "splits" in p.lower()), None)
assert SPLITS, "attach bn-asr-splits"
TGT = next((f"{p}/task_b_targets_n100.csv" for p in INPUTS_LIST
            if os.path.exists(f"{p}/task_b_targets_n100.csv")), None)
assert TGT, "attach one eps50-*-n100 dataset for the frozen 100-utterance list"
SHA = hashlib.sha256(open(TGT, "rb").read()).hexdigest()
assert SHA == "8711dc56202b33f90e769c9735231b0fa8cf27fcbc20151f5466bcd9cc2b1256", (
    "not the frozen target file the attacks used; the WER would not match the set")
print("targets:", TGT, "\nsha256 :", SHA)

ZW = dict.fromkeys(map(ord, "‌‍"), None)
def norm_bn(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).translate(ZW)).strip()

def fix_paths(df):
    p0 = str(df["path"].iloc[0])
    if os.path.exists(p0): return df
    stem, ext = os.path.basename(os.path.dirname(p0)), os.path.splitext(p0)[1]
    c = [d for pat in (f"/kaggle/input/*/{stem}", f"/kaggle/input/*/*/{stem}",
                       f"/kaggle/input/*/*/*/{stem}")
         for d in glob.glob(pat) if os.path.isdir(d)]
    assert c, f"audio dir '{stem}' not found -- attach bengaliai-speech"
    df = df.copy(); df["path"] = c[0] + "/" + df["utt_id"].astype(str) + ext
    return df

tg = pd.read_csv(TGT)
assert len(tg) == 100 and tg.utt_id.nunique() == 100
_full = pd.read_csv(f"{SPLITS}/test.csv")
attack = tg[["utt_id"]].merge(_full, on="utt_id", how="left")
assert attack["path"].notna().all(), "some of the 100 are absent from test.csv"
assert (attack.text.map(norm_bn).values == tg.text.map(norm_bn).values).all(), \
    "transcript mismatch between the target file and test.csv"
attack = fix_paths(attack)
N25 = 25   # manifest order: the first 25 are the published RQ1 spine
print(f"attack set: {len(attack)} utterances, first {N25} = the RQ1 spine")

## 5 · Transcribe and score

In [ ]:
def load_wavs(paths):
    out = []
    for p in paths:
        try:
            a, _ = librosa.load(p, sr=16000, mono=True, duration=MAX_DUR)
            a = np.asarray(a, np.float32)
            pk = np.abs(a).max()
            out.append(a / pk if pk > 1.0 else a)
        except Exception:
            out.append(np.zeros(16000, np.float32))
    return out


@torch.no_grad()
def transcribe(model, processor, wavs, bs=EVAL_BS):
    """Length-sorted batched greedy CTC decode. Order is restored at the end."""
    order = np.argsort([len(w) for w in wavs])
    hyps = [None] * len(wavs)
    do_norm = bool(getattr(processor.feature_extractor, "do_normalize", True))
    for i in range(0, len(order), bs):
        idx = order[i:i+bs]
        chunk = [wavs[j] for j in idx]
        T = max(len(w) for w in chunk)
        x = torch.zeros(len(chunk), T, device=DEVICE)
        m = torch.zeros(len(chunk), T, device=DEVICE)
        for k, w in enumerate(chunk):
            x[k, :len(w)] = torch.from_numpy(w).to(DEVICE); m[k, :len(w)] = 1
        if do_norm:
            n = m.sum(-1, keepdim=True).clamp(min=1)
            mu = (x*m).sum(-1, keepdim=True)/n
            var = (((x-mu)*m)**2).sum(-1, keepdim=True)/n
            x = ((x-mu)/torch.sqrt(var+1e-7))*m
        ids = torch.argmax(model(input_values=x, attention_mask=m.long()).logits, -1)
        for k, j in enumerate(idx):
            hyps[j] = norm_bn(processor.batch_decode(ids[k:k+1])[0])
    return hyps


def _edit(r, h):
    """Levenshtein on token lists."""
    if not r: return len(h)
    d = list(range(len(h) + 1))
    for i in range(1, len(r) + 1):
        prev, d[0] = d[0], i
        for j in range(1, len(h) + 1):
            cur = d[j]
            d[j] = min(d[j] + 1, d[j-1] + 1, prev + (r[i-1] != h[j-1]))
            prev = cur
    return d[len(h)]

def score(refs, hyps):
    """CORPUS-level WER and CER: total edits / total reference length.

    Implemented inline rather than with jiwer, because `jiwer` is NOT preinstalled
    on Kaggle -- it was supplied by the pip cell, which cannot run with internet
    off. Verified against jiwer 4.x to machine precision on light/heavy/large
    corpora and on both degenerate cases (empty hypothesis -> 1.0, over-long
    hypothesis -> 1.5, i.e. WER may exceed 1.0).

    Corpus-level, NOT the mean of per-utterance rates. jiwer.wer(list, list)
    aggregates, so every published figure in this project is an aggregate, and a
    per-utterance mean would silently differ.
    """
    pairs = [(norm_bn(r), h) for r, h in zip(refs, hyps) if str(r).strip()]
    if not pairs: return 1.0, 1.0
    R = [p[0] for p in pairs]; H = [p[1] for p in pairs]
    wn = sum(_edit(r.split(), h.split()) for r, h in zip(R, H))
    wd = sum(len(r.split()) for r in R)
    cn = sum(_edit(list(r), list(h)) for r, h in zip(R, H))
    cd = sum(len(r) for r in R)
    return (wn / wd if wd else 1.0), (cn / cd if cd else 1.0)


print("loading audio...")
t0 = time.time()
ATTACK_WAVS = load_wavs(attack["path"].tolist())
print(f"  {len(ATTACK_WAVS)} clips in {time.time()-t0:.0f}s")

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
t0 = time.time()
proc = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
mdl = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR).float().to(DEVICE).eval()
mdl.config.apply_spec_augment = False
hyp = transcribe(mdl, proc, ATTACK_WAVS)
refs = list(attack["text"])
w100, c100 = score(refs, hyp)
w25,  c25  = score(refs[:N25], hyp[:N25])
print(f"transcribed {len(hyp)} clips in {time.time()-t0:.0f}s\n")
print("=" * 66)
print(f"  {CKPT_DATASET}/{CHECKPOINT}")
print("=" * 66)
print(f"  clean WER, all 100       {w100:.4f}      <- THE COVARIATE")
print(f"  clean CER, all 100       {c100:.4f}")
print(f"  clean WER, original 25   {w25:.4f}")
print(f"  clean CER, original 25   {c25:.4f}")
print(f"\n  for comparison, ckpt-xlsr-1h/final (the FIRST training run):")
print(f"    all 100  0.6355      original 25  0.6471")
print(f"  and the 25-utt figure previously recorded for this traj model: 0.6716")
print(f"\n  difference in clean WER between the two 1 h training runs, all 100: "
      f"{w100-0.6355:+.4f}")
print(f"  difference in ATTACK success between them: 41.0% vs 75.0% = -34.0 pp")
print("\n  If clean WER is close between the two runs but attack success differs by 34 pp,")
print("  that is the point: accuracy matching does NOT pin down adversarial robustness.")

out = pd.DataFrame([dict(model=CKPT_DATASET, checkpoint=CHECKPOINT, model_dir=MODEL_DIR,
                         n=len(hyp), wer_attack100=w100, cer_attack100=c100,
                         wer_attack25=w25, cer_attack25=c25)])
out.to_csv(f"{WORK}/clean_wer_traj.csv", index=False)
import transformers
json.dump(dict(notebook="NB-02c-clean-wer-traj", model_dir=MODEL_DIR, n=len(hyp),
               wer_attack100=w100, cer_attack100=c100, wer_attack25=w25, cer_attack25=c25,
               targets_sha256=SHA, seed=SEED, torch=torch.__version__,
               transformers=transformers.__version__,
               gpu=torch.cuda.get_device_name(0),
               timestamp=time.strftime("%Y-%m-%d %H:%M:%S")),
          open(f"{WORK}/_provenance.json", "w"), indent=2)
json.dump({"title": "clean-wer-traj", "id": "YOURUSERNAME/clean-wer-traj",
           "licenses": [{"name": "CC0-1.0"}]},
          open(f"{WORK}/dataset-metadata.json", "w"), indent=2)
print("\nfiles:", sorted(os.listdir(WORK)))
print("Send me clean_wer_traj.csv -- one number, wer_attack100.")

---

Send me `wer_attack100`. It goes straight into the pooled ANCOVA as the seventh condition's
covariate, and it decides whether the central interval excludes zero or not.